In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Any, Dict, Optional, Set

import numpy as np
import pandas as pd
from pymongo import MongoClient
import yaml


In [ ]:
def load_config(path: str | Path = "../config/config.yaml") -> dict[str, Any]:
    path = Path(path)
    if not path.exists():
        path = Path("config/config.yaml")
    with path.open("r", encoding="utf-8") as f:
        return yaml.safe_load(f)


config = load_config()
mongo_config = config["mongo"]

In [ ]:
client = MongoClient(mongo_config.get('url'))
db = client[mongo_config.get('db')]
col_schedule = db[mongo_config.get('collection').get('collection_schedule')]
season = '2010-2011'
df_schedule = pd.DataFrame(col_schedule.find({'season': season}))
check_games = df_schedule['game_id'].sample(10, random_state=1909).values.tolist()

In [ ]:
col_team_stats = db[mongo_config.get('collection').get('collection_team_game_stats')]
col_player_stats = db[mongo_config.get('collection').get('collection_player_game_stats')]

df_team_stats = pd.DataFrame(col_team_stats.find({'game_id': {"$in": check_games}}))
df_player_stast = pd.DataFrame(col_player_stats.find({'game_id': {"$in": check_games}}))

In [ ]:
pd.set_option("display.max_rows", None)

for game in check_games:
    team_stats = df_team_stats[df_team_stats.game_id == game]
    team_stats = team_stats.sort_values(by='game_venue', ascending=False).reset_index(drop=True)
    game_info = team_stats[['game_id', 'game_date', 'week', 'team_name', 'game_venue']]
    stats = pd.json_normalize(team_stats.stats)
    stats_df = stats[[col for col in stats.columns if 'ft.' in col]]
    
    final_game = pd.merge(
        game_info,
        stats_df,
        left_index=True,
        right_index=True
    )
    display(final_game.T)
    print("\n" + "-" * 80 + "\n")


In [ ]:
mongo_config

In [ ]:
# db[mongo_config.get('collection').get("collection_schedule")].delete_many({})
# db[mongo_config.get('collection').get("collection_raw_events")].delete_many({})
# db[mongo_config.get('collection').get("collection_processed_events")].delete_many({})
# db[mongo_config.get('collection').get("collection_team_game_stats")].delete_many({})
# db[mongo_config.get('collection').get("collection_player_game_stats")].delete_many({})

DeleteResult({'n': 0, 'ok': 1.0}, acknowledged=True)